# ecgdig híbrido: clássico × pesos do 5º colocado

Experimento isolado no branch **hibrido**. Este notebook apenas lê o GitHub; não faz commits nem altera o `main`.

1. Selecione **Ambiente de execução → Alterar tipo de ambiente → GPU T4** (ou outra GPU CUDA).
2. Como o repositório é privado, adicione um segredo `GITHUB_TOKEN` no ícone de chave do Colab, com acesso de **leitura de conteúdo apenas a este repositório**, e habilite seu acesso pelo notebook. Sem segredo, a célula solicitará o token em campo oculto.
3. Use **Executar tudo**. Os pesos públicos são baixados sem token do Kaggle.

O piloto compara 12 imagens sintéticas em seis formatos. Não faz fine-tuning. A grade do quarto canal é anônima, portanto esta é uma avaliação de transferência, não reprodução do placar do Kaggle. Resultados locais preliminares não são reutilizados aqui.


In [ ]:
# Configuração
REPOSITORY = "fmenegui/ecg-digitalization-classic"
BRANCH = "hibrido"
UPSTREAM_COMMIT = "8358d5b09a5b9519275e9b54c11716da5752a9d6"
LIMIT = 12  # piloto completo; não aumente sem acrescentar casos ao manifesto
SAVE_TO_DRIVE = False  # opcional: pede autorização do Google quando True
DOWNLOAD_ZIP = True

from pathlib import Path
ROOT = Path("/content/ecgdig-hibrido")
UPSTREAM = Path("/content/kaggle-5th")


In [ ]:
# Baixar uma revisão fixa do branch privado, sem colocar o token na URL ou nos logs.
import io, tarfile, getpass, requests
try:
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
except Exception:
    token = getpass.getpass("Token GitHub de leitura (campo oculto): ")
if not token:
    raise RuntimeError("É necessário acesso de leitura ao repositório privado.")
headers = {"Authorization": f"Bearer {token}", "Accept": "application/vnd.github+json"}
response = requests.get(f"https://api.github.com/repos/{REPOSITORY}/commits/{BRANCH}", headers=headers, timeout=60)
response.raise_for_status()
EXPERIMENT_COMMIT = response.json()["sha"]
response = requests.get(f"https://api.github.com/repos/{REPOSITORY}/tarball/{EXPERIMENT_COMMIT}", headers=headers, timeout=180)
response.raise_for_status()
ROOT.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(response.content), mode="r:gz") as archive:
    for member in archive.getmembers():
        relative = Path(*Path(member.name).parts[1:])
        if relative.is_absolute() or ".." in relative.parts:
            raise RuntimeError("Caminho inválido no arquivo GitHub")
        if member.isfile():
            target = ROOT / relative
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.extractfile(member).read())
del token, headers, response
print("Revisão do experimento:", EXPERIMENT_COMMIT)


In [ ]:
# Dependências; preserva o PyTorch CUDA fornecido pelo Colab.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "timm==1.0.31", "hydra-core>=1.3", "transformers>=4.48", "einops",
                "opencv-python-headless<5", "numpy<2.3", "pytest", "matplotlib",
                "-e", str(ROOT)], check=True)
if not UPSTREAM.exists():
    subprocess.run(["git", "init", "-q", str(UPSTREAM)], check=True)
subprocess.run(["git", "-C", str(UPSTREAM), "fetch", "--depth", "1",
                "https://github.com/dangnh0611/kaggle_ecg_digitization.git", UPSTREAM_COMMIT], check=True)
subprocess.run(["git", "-C", str(UPSTREAM), "checkout", "--detach", "FETCH_HEAD"], check=True)


In [ ]:
# Verificação da GPU em um processo novo, depois da instalação.
import json
probe = subprocess.run([sys.executable, "-c", "import torch,json; print(json.dumps({'cuda':torch.cuda.is_available(),'torch':torch.__version__,'gpu':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}))"], capture_output=True, text=True, check=True)
GPU = json.loads(probe.stdout.strip().splitlines()[-1])
print(GPU)
if not GPU["cuda"]:
    raise RuntimeError("Ative uma GPU no menu de ambiente de execução e execute novamente.")


In [ ]:
# Colab tem espaço para baixar o checkpoint completo: evita depender de HTTP Range.
MODEL = ROOT / "models/kaggle5"
required = [MODEL / name for name in ("ema.pt", "config.yaml", "provenance.json")]
if not all(path.exists() for path in required):
    print("Baixando checkpoint oficial (~941 MB); somente os pesos EMA serão mantidos.", flush=True)
    result = subprocess.run([sys.executable, "hybrid/download_kaggle5.py", "--out", str(MODEL), "--mode", "full"],
                            cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(result.stdout)
    if result.returncode:
        raise RuntimeError("Downloader falhou. Diagnóstico completo:\n" + result.stdout[-6000:])
print((MODEL / "provenance.json").read_text())


## Testes de integração

Estes testes verificam escalas dos recortes, equivalência das operações locais e preservação de layout/calibração. A suíte sintética antiga tem uma asserção de superioridade do campo por células que falhou no ambiente local; ela não é escondida nem usada para ajustar o resultado deste piloto.


In [ ]:
import os
ENV = dict(os.environ)
ENV["PYTHONPATH"] = os.pathsep.join([str(UPSTREAM / "src"), str(UPSTREAM / "yagm/src"), str(ROOT)])
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_hybrid.py"], cwd=ROOT, env=ENV, check=True)


## Comparação pareada

Cada imagem passa pelo clássico e pelo híbrido. Layout, ganho, velocidade e duração devem ser idênticos. O SNR penaliza amostras ausentes como zero, sem alinhamento posterior. O modelo é carregado com correspondência estrita dos pesos, em FP16. Em CUDA, a rede usa o decoder integral; as faixas de economia de memória ficam restritas a CPU/Metal.

A execução salva cada resultado concluído e pode ser retomada **no mesmo diretório, sem mudar a revisão ou o protocolo**. Escolha uma nova pasta se trocar pesos, versão ou casos.


In [ ]:
OUT = ROOT / "results" / f"colab-{EXPERIMENT_COMMIT[:10]}"
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "run_source.json").write_text(json.dumps({"repository": REPOSITORY, "branch": BRANCH, "commit": EXPERIMENT_COMMIT, "upstream_commit": UPSTREAM_COMMIT, "gpu": GPU}, indent=2))
subprocess.run([sys.executable, "-m", "hybrid.compare_kaggle5", "--source", str(UPSTREAM),
                "--model", str(MODEL), "--device", "cuda", "--limit", str(LIMIT), "--out", str(OUT)],
               cwd=ROOT, env=ENV, check=True)
subprocess.run([sys.executable, "hybrid/plot_pilot.py", str(OUT)], cwd=ROOT, check=True)


In [ ]:
# Relatório por formato e figura; os CSVs contêm cada imagem e variante.
import pandas as pd
from IPython.display import display, HTML, Image
summary = json.loads((OUT / "summary.json").read_text())
display(pd.DataFrame(summary["formats"]))
display(pd.read_csv(OUT / "paired.csv"))
display(Image(filename=str(OUT / "comparison.png")))
display(HTML((OUT / "report.html").read_text()))


In [ ]:
# Exportar resultados, sinais e manifesto; os pesos não entram no ZIP.
import shutil
ZIP_PATH = Path(shutil.make_archive(str(OUT), "zip", root_dir=OUT))
print("Arquivo:", ZIP_PATH)
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    destination = Path("/content/drive/MyDrive/ecgdig-experimentos")
    destination.mkdir(parents=True, exist_ok=True)
    shutil.copy2(ZIP_PATH, destination / ZIP_PATH.name)
    print("Salvo no Drive:", destination / ZIP_PATH.name)
if DOWNLOAD_ZIP:
    from google.colab import files
    files.download(str(ZIP_PATH))


## Como interpretar

- Compare `delta` por imagem e formato; um ganho médio pode esconder perdas graves.
- Verifique cobertura, RMSE, erro nos picos e tempo em `per_image.csv`.
- A mudança de template, ganho, velocidade e renderizador pode afetar a rede mesmo com recortes locais.
- Dois casos por formato não demonstram universalidade. Se a transferência falhar, o próximo passo é ajuste multiformato ou destilação para um segmentador local menor.
- Este notebook não treina, não escolhe a melhor variante pelo gabarito e não escreve no GitHub.
